# 03 — Seasonal Decomposition — Spain CPI

Additive, multiplicative, and STL decomposition of the Spain CPI index (monthly, 2002–2024). Quantifies seasonal strength (Fs metric) to justify the seasonal differencing order D for SARIMA.

**Input:** `data/processed/ipc_spain_index.parquet`  
**Establishes:** Seasonal strength (Fs), preferred decomposition, monthly pattern

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose, STL

ROOT = Path('..').resolve()         # tfg-forecasting/
MONOREPO = ROOT.parent              # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))

from shared.constants import DATE_TRAIN_END

plt.rcParams.update({"figure.figsize": (14, 8), "axes.grid": True, "grid.alpha": 0.3})

In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "ipc_spain_index.parquet")
train = df.loc[:DATE_TRAIN_END]
y = train["indice_general"]
y.index.freq = "MS"
print(f"Train: {len(y)} obs, {y.index.min().date()} -> {y.index.max().date()}")

Train: 228 obs, 2002-01-01 -> 2020-12-01


## 1. Additive Decomposition

In [ ]:
decomp_add = seasonal_decompose(y, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp_add.observed.plot(ax=axes[0], title="Observed")
decomp_add.trend.plot(ax=axes[1], title="Trend")
decomp_add.seasonal.plot(ax=axes[2], title="Seasonality")
decomp_add.resid.plot(ax=axes[3], title="Residual")
fig.suptitle("Classical Additive Decomposition (period=12)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2. Multiplicative Decomposition

In [ ]:
decomp_mul = seasonal_decompose(y, model="multiplicative", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp_mul.observed.plot(ax=axes[0], title="Observed")
decomp_mul.trend.plot(ax=axes[1], title="Trend")
decomp_mul.seasonal.plot(ax=axes[2], title="Seasonality (factor)")
decomp_mul.resid.plot(ax=axes[3], title="Residual")
fig.suptitle("Classical Multiplicative Decomposition (period=12)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. STL Decomposition

In [ ]:
stl = STL(y, period=12, robust=True)
stl_result = stl.fit()

fig = stl_result.plot()
fig.set_size_inches(14, 10)
fig.suptitle("STL Decomposition (LOESS, robust)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Seasonal Strength (Fs)

Metric by Wang, Smith & Hyndman (2006):

$F_s = 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}$

- $F_s > 0.64$ indicates significant seasonality
- $F_s$ close to 1 indicates very strong seasonality

In [ ]:
seasonal = stl_result.seasonal
resid = stl_result.resid

Fs = 1 - np.var(resid) / np.var(seasonal + resid)
print(f"Seasonal strength (Fs): {Fs:.4f}")
print(f"Seasonality: {'SIGNIFICANT' if Fs > 0.64 else 'NOT significant'} (threshold 0.64)")

## 5. Monthly Seasonality Pattern

In [ ]:
seasonal_pattern = stl_result.seasonal.groupby(stl_result.seasonal.index.month).mean()

fig, ax = plt.subplots(figsize=(10, 5))
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
colors = ["#d62728" if v > 0 else "#1f77b4" for v in seasonal_pattern.values]
ax.bar(range(1, 13), seasonal_pattern.values, color=colors)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_title("Average Monthly Seasonal Component (STL)")
ax.set_ylabel("Seasonal effect (index points)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("Monthly seasonal pattern:")
for m, name in enumerate(months, 1):
    print(f"  {name}: {seasonal_pattern[m]:+.3f}")

## 6. Temporal Stability

In [ ]:
seasonal_by_year = stl_result.seasonal.copy()
seasonal_by_year = pd.DataFrame({
    "year": seasonal_by_year.index.year,
    "month": seasonal_by_year.index.month,
    "seasonal": seasonal_by_year.values,
})

pivot = seasonal_by_year.pivot(index="month", columns="year", values="seasonal")

fig, ax = plt.subplots(figsize=(14, 6))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], alpha=0.3, linewidth=0.8, color="grey")

# Superimposed mean
ax.plot(pivot.index, pivot.mean(axis=1), linewidth=2.5, color="#d62728", label="Mean")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_title("Seasonal Pattern Stability over the Years")
ax.set_ylabel("Seasonal component")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Conclusion

**Key results:**
- STL decomposition shows increasing trend with acceleration post-2021
- Fs quantifies seasonal strength
- The seasonal pattern identifies months with the largest/smallest effects
- If Fs > 0.64, seasonal differencing (D=1, m=12) is justified for SARIMA

*Fill with actual values after execution.*